# Hybrid Retrieval Starter

Use this notebook to prototype the retrieval layer before wiring it into FastAPI. The first pass runs BM25 locally over the existing NED PDFs. Dense retrieval uses the existing Pinecone setup when `PINECONE_API_KEY` and an OpenAI key are available.

In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
print(PROJECT_ROOT)
print(DATA_DIR)

In [ ]:
from modules.hybrid_retriever import (
    BM25Retriever,
    CrossEncoderReranker,
    HybridRetriever,
    get_pinecone_vectorstore,
)

bm25 = BM25Retriever.from_pdf_dir(str(DATA_DIR))
print(f"BM25 corpus chunks: {len(bm25.documents)}")

In [ ]:
query = "What are the admission requirements for undergraduate engineering programs?"

bm25_results = bm25.search(query, top_k=5)

for result in bm25_results:
    meta = result.document.metadata
    print(f"rank={result.rank} score={result.score:.3f} source={meta.get('source')} page={meta.get('page')} chunk={meta.get('chunk_id')}")
    print(result.document.page_content[:350].replace("\n", " "))
    print()

In [ ]:
has_pinecone = bool(os.getenv("PINECONE_API_KEY"))
has_openai = bool(os.getenv("OPENAI_API_KEY") or os.getenv("OPEN_AI_KEY"))

dense_vectorstore = get_pinecone_vectorstore() if has_pinecone and has_openai else None
print("Dense retrieval enabled:", dense_vectorstore is not None)

hybrid = HybridRetriever(
    bm25_retriever=bm25,
    dense_vectorstore=dense_vectorstore,
    reranker=None,
    bm25_k=8,
    dense_k=8,
    fusion_k=10,
    final_k=4,
)

docs = hybrid.retrieve(query)
for i, doc in enumerate(docs, start=1):
    meta = doc.metadata
    print(f"{i}. source={meta.get('source')} page={meta.get('page')} chunk={meta.get('chunk_id')} rrf={meta.get('rrf_score'):.4f}")
    print("ranks:", meta.get("retrieval_ranks"))
    print(doc.page_content[:350].replace("\n", " "))
    print()

In [ ]:
# Turn this on after installing sentence-transformers.
ENABLE_RERANKER = False

if ENABLE_RERANKER:
    hybrid.reranker = CrossEncoderReranker()
    reranked_docs = hybrid.retrieve(query)
    for i, doc in enumerate(reranked_docs, start=1):
        meta = doc.metadata
        print(f"{i}. reranker={meta.get('reranker_score'):.4f} source={meta.get('source')} page={meta.get('page')}")
        print(doc.page_content[:350].replace("\n", " "))
        print()